In [ ]:
import time

import pandas as pd
import numpy as np

from nixtla import NixtlaClient

from utilsforecast.losses import mae
from utilsforecast.evaluation import evaluate
import os
from dotenv import load_dotenv

load_dotenv()

In [ ]:
nixtla_client = NixtlaClient(
    base_url=os.environ["TIMEGEN_ENDPOINT"] ,
    api_key=os.environ["TIMEGEN_KEY"] ,
)

In [ ]:

df = pd.read_csv(
    "https://raw.githubusercontent.com/Nixtla/transfer-learning-time-series/main/datasets/m5_sales_exog_small.csv"
)
df["ds"] = pd.to_datetime(df["ds"])

df.head()

nixtla_client.plot(
    df,
    max_insample_length=365,
)


df_transformed = df.copy()

df_transformed["y"] = np.log(df_transformed["y"] + 1)

df_transformed.head()



test_df = df_transformed.groupby("unique_id").tail(28)

input_df = df_transformed.drop(test_df.index).reset_index(drop=True)

In [ ]:

start = time.time()

fcst_df = nixtla_client.forecast(
    df=input_df,
    h=28,
    level=[80],  # Generate a 80% confidence interval
    finetune_steps=10,  # Specify the number of steps for fine-tuning
    finetune_loss="mae",  # Use the MAE as the loss function for fine-tuning
    time_col="ds",
    target_col="y",
    id_col="unique_id",
)

end = time.time()

TimeGEN_duration = end - start

print(f"Time (TimeGEN): {TimeGEN_duration}")

In [ ]:
cols = [col for col in fcst_df.columns if col not in ["ds", "unique_id"]]

for col in cols:
    fcst_df[col] = np.exp(fcst_df[col]) - 1

fcst_df.head()

evaluation

In [ ]:

nixtla_client.plot(
    test_df, fcst_df, models=["TimeGPT"], level=[80], time_col="ds", target_col="y"
)

fcst_df["ds"] = pd.to_datetime(fcst_df["ds"])

test_df = pd.merge(test_df, fcst_df, "left", ["unique_id", "ds"])
evaluation = evaluate(
    test_df, metrics=[mae], models=["TimeGPT"], target_col="y", id_col="unique_id"
)

average_metrics = evaluation.groupby("metric")["TimeGPT"].mean()
average_metrics